In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import re
import string
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

In [4]:
# Define pseudocolor mapping rules - helpful to generate images on the fly
pseudocolor_map = {
    "DAPI": "blue",
    "Brightfield": "gray",
    "Alexa 488": "green",
    "Alexa 568": "red",
    "Alexa 647": "magenta"
}

# Define pseudocolor mapping rules for matplotlib - helpful to generate images on the fly
mpl_colormaps = {
    "blue": LinearSegmentedColormap.from_list("black_blue", [(0, 0, 0), (0, 0, 1)]),
    "green": LinearSegmentedColormap.from_list("black_green", [(0, 0, 0), (0, 1, 0)]),
    "red": LinearSegmentedColormap.from_list("black_red", [(0, 0, 0), (1, 0, 0)]),
    "magenta": LinearSegmentedColormap.from_list("black_magenta", [(0, 0, 0), (1, 0, 1)]),
    "gray": LinearSegmentedColormap.from_list("black_gray", [(0, 0, 0), (1, 1, 1)])
}

In [2]:
plateID_to_folder_ID_fn = "/vf/users/CARDPB2/iNDI/Production/metadata/indi_plateID_to_folderID.csv"
plateID_to_folder_ID = pd.read_csv(plateID_to_folder_ID_fn)
plateID_to_folder_ID

,iNDI_Plate_ID,Layout,Folder_ID
0,INDI00002D,Plate002,2b1c31ad-579d-4ea6-a782-251ea083ed6a
1,INDI00007D,Plate007,d7e04b5c-a253-42ff-859f-2af0e6047a00
2,INDI00011D,Plate011,f8062781-891b-4688-a59e-fccbf48325cf
3,INDI00015D,Plate015,caef83c7-6b44-4b64-b8ec-e8d84ca6759d
4,INDI00019D,Plate019,c8f2dbbe-6632-4063-ab68-0b4fa37909af
5,INDI00023D,Plate023,32d75d22-bd17-4629-8257-a4056e3ffcee
6,INDI00026D,Plate026,c6a92be6-a13a-4957-a62d-9f99961c903f
7,INDI00035D,Plate002,c170959e-633a-458c-8542-2aa619c0d7f8
8,INDI00039D,Plate039,07fc5da6-9d7d-4c97-858b-4b76df1859a5
9,INDI00043D,Plate043,4405a3b2-6b88-49b1-91f3-992e09ccbd16


In [6]:
# Namespace consistent across all experiments
NS = {'h': '43B2A954-E3C3-47E1-B392-6635266B0DD3/HarmonyV7'}
BASE_PATH = Path("/data/CARDPB2/iNDI/Production")

# XML helpers
def _find_text(el, tag):
    """Return text of a matched subelement, or None if not found."""
    match = el.find(tag, NS)
    return match.text if match is not None else None


def parse_experiment_xml(experiment_path: Path) -> dict:
    """Extract measurement ID, date, and plate name from the experiment XML."""
    xml_file = next(experiment_path.glob("*.xml"), None)
    if xml_file is None:
        raise FileNotFoundError(f"No experiment XML found in {experiment_path}")

    root = ET.parse(xml_file).getroot()
    return {
        "measurement_id": _find_text(root, 'h:MeasurementID'),
        "date":           _find_text(root, 'h:Date'),
        "plate":          _find_text(root, 'h:InitialPlateName'),
    }


def parse_index_xml(experiment_path: Path) -> dict:
    """Extract plate ID, resolution, and channel info from the index XML."""
    index_xml = next((experiment_path / "index").glob("*.xml"), None)
    if index_xml is None:
        raise FileNotFoundError(f"No index XML found in {experiment_path / 'index'}")

    root = ET.parse(index_xml).getroot()

    plate_id = _find_text(root, './/h:PlateID')
    x_res = float(_find_text(root, './/h:ImageResolutionX')) * 1e6
    y_res = float(_find_text(root, './/h:ImageResolutionY')) * 1e6
    channel_df = _parse_channels(root)

    return {"plate_id": plate_id, "x_res": x_res, "y_res": y_res, "channel_df": channel_df}


def _parse_channels(root) -> pd.DataFrame:
    """Extract channel metadata from the index XML root into a DataFrame."""
    channels = []
    for map_el in root.findall(".//h:Map", NS):
        first_entry = map_el.find("h:Entry", NS)
        if first_entry is None or first_entry.find("h:ChannelName", NS) is None:
            continue
        for entry in map_el.findall("h:Entry", NS):
            ch_id = entry.attrib.get("ChannelID")
            channels.append({
                "ChannelID":     int(ch_id) if ch_id is not None else None,
                "Channel_name":  _find_text(entry, "h:ChannelName"),
                "Type":          _find_text(entry, "h:ChannelType"),
                "Excitation_nm": _find_text(entry, "h:MainExcitationWavelength"),
                "Emission_nm":   _find_text(entry, "h:MainEmissionWavelength"),
            })
        break  # Only process the first matching Map

    return pd.DataFrame(channels).sort_values("ChannelID").reset_index(drop=True)


# Image file parsing
def _parse_filename(name: str) -> list:
    """Parse r/c/f/p/ch/t components from an Opera Phenix TIFF filename."""
    match = re.match(r"r(\d+)c(\d+)f(\d+)p(\d+)-ch(\d+)t(\d+)", name)
    return [int(g) for g in match.groups()] if match else [None] * 6


def build_image_df(img_dir: Path) -> pd.DataFrame:
    """Collect all TIFFs under img_dir and parse their filename metadata."""
    files = sorted(f for f in img_dir.rglob("*") if f.suffix.lower() == ".tiff")
    df = pd.DataFrame({
        "filepath":     files,
        "filename":     [f.name for f in files],
        "subdirectory": [f.parent.relative_to(img_dir) for f in files],
    })
    df[["Row", "Column", "Frame", "Plane", "ChannelID", "Time"]] = (
        df["filename"].apply(lambda x: pd.Series(_parse_filename(x)))
    )
    return df


# Top-level loader 
def load_experiment(experiment_name: str, pseudocolor_map: dict, mpl_colormaps: dict) -> dict:
    """
    Load all metadata and image file info for a given experiment.

    Parameters
    ----------
    experiment_name : str
        Experiment UUID / folder name under BASE_PATH.
    pseudocolor_map : dict
        Maps channel names to pseudocolor strings (e.g. {"DAPI": "blue"}).
    mpl_colormaps : dict
        Maps pseudocolor strings to matplotlib colormap names.

    Returns
    -------
    dict with keys: experiment_meta, index_meta, image_df, merged_df, summary
    """
    experiment_path = BASE_PATH / experiment_name

    experiment_meta = parse_experiment_xml(experiment_path)
    index_meta = parse_index_xml(experiment_path)

    channel_df = index_meta["channel_df"].copy()
    channel_df["Pseudocolor"]    = channel_df["Channel_name"].map(pseudocolor_map).fillna("gray")
    channel_df["MPL_colormap"]   = channel_df["Pseudocolor"].str.lower().map(mpl_colormaps)
    channel_df["Measurement_ID"] = experiment_meta["measurement_id"]
    channel_df["Measurement_date"] = experiment_meta["date"]
    channel_df["Plate_ID"]       = index_meta["plate_id"]
    channel_df["res_x"]          = index_meta["x_res"]
    channel_df["res_y"]          = index_meta["y_res"]

    img_dir = experiment_path / "images"
    image_df = build_image_df(img_dir)
    merged_df = pd.merge(image_df, channel_df, on="ChannelID")

    summary = {
        "measurement_id": experiment_meta["measurement_id"],
        "plate":          experiment_meta["plate"],
        "wells":          merged_df[["Row", "Column"]].drop_duplicates().shape[0],
        "channels":       merged_df["ChannelID"].nunique(),
        "z_planes":       merged_df["Plane"].nunique(),
        "frames":         merged_df["Frame"].nunique(),
        "timepoints":     merged_df["Time"].nunique(),
    }

    print(f"""
        Experiment ID:       {summary['measurement_id']}
        Plate ID:            {summary['plate']}
        Wells imaged:        {summary['wells']}
        Frames per well:     {summary['frames']}
        Channels per image:  {summary['channels']}
        Z-slices per image:  {summary['z_planes']}
        Timepoints:          {summary['timepoints']}
        """)

    return {
        "experiment_meta": experiment_meta,
        "index_meta":      index_meta,
        "channel_df":      channel_df,
        "image_df":        image_df,
        "merged_df":       merged_df,
        "summary":         summary,
    }